# Workshop: Gateway Integration for SRE Agent

## Overview

In this workshop module, you'll enhance your SRE agent with Amazon Bedrock AgentCore Gateway integration, transforming your tools into secure production-ready services using the Model Context Protocol (MCP).

### Learning Objectives

By the end of this module, you will:
- Understand AgentCore Gateway architecture and security benefits
- Configure a Gateway with MCP protocol integration
- Implement OAuth authentication for secure access
- Transform existing tools to use the Gateway
- Run secure investigations through the Gateway

### Prerequisites

- AWS Account with Amazon Bedrock access
- Claude 3 Haiku model enabled in your AWS account
- Python 3.9+ environment
- Completion of Module 1 (Multiple Tools Agent)
- AWS CLI configured with appropriate permissions

### Architecture

```
┌─────────────────┐    ┌─────────────────┐    ┌─────────────────┐    ┌─────────────────┐
│                 │    │                 │    │                 │    │                 │
│ Strands Agent   │───▶│ AgentCore       │───▶│ MCP Protocol    │───▶│ Backend APIs    │
│                 │    │ Gateway         │    │ Tools           │    │                 │
│ • Claude Haiku  │    │ • Authentication│    │ • Pod Status    │    │ • Pod Data      │
│ • Investigation │    │ • Security      │    │ • Pod Events    │    │ • Events        │
│ • Orchestration │    │ • Monitoring    │    │ • Pod Resources │    │ • Resources     │
└─────────────────┘    └─────────────────┘    └─────────────────┘    └─────────────────┘
```

**Estimated completion time:** 45 minutes

## Step 1: Environment Setup

Install the required packages for this workshop module.

In [ ]:
%%bash
pip install fastapi uvicorn strands requests boto3 oauthlib pyyaml --quiet
echo "✅ Packages installed successfully"

In [ ]:
# Import required libraries
from fastapi import FastAPI, HTTPException, Query, Depends, Security, status
from fastapi.security import OAuth2PasswordBearer, OAuth2PasswordRequestForm
from strands import Agent, tool
from strands.models import BedrockModel
from typing import List, Dict, Any, Optional
import uvicorn
import threading
import time
import requests
import boto3
import os
import json
import yaml
import uuid
import logging
from datetime import datetime, timedelta

print("✅ Libraries imported successfully")

## Step 2: Understanding Gateway Architecture

Before implementing, let's understand the key components of the AgentCore Gateway architecture and how it enhances security and scalability.

### Gateway Architecture Overview

Amazon Bedrock AgentCore Gateway provides a secure intermediary between your agent and backend services:

**Key Components:**

1. **Authentication Layer**
   - OAuth 2.0 authentication flow
   - Token validation and management
   - Role-based access control

2. **MCP Protocol Layer**
   - Standardized Model Context Protocol
   - Tool definition and routing
   - Request/response transformation

3. **Security Features**
   - HTTPS/TLS encryption
   - Input validation
   - Rate limiting and throttling

4. **Observability**
   - Request logging
   - Performance monitoring
   - Usage analytics

**Benefits of Gateway Architecture:**

- **Enhanced Security**: Authentication, encryption, and access control
- **Standardization**: Consistent tool interfaces via MCP protocol
- **Scalability**: Centralized management of multiple backends
- **Monitoring**: Comprehensive visibility into tool usage
- **Production Readiness**: Enterprise-grade security and reliability

In this workshop, we'll transition from direct API calls to a gateway-mediated architecture for production-ready security.

## Step 3: Create Backend Services

We'll reuse our enhanced FastAPI backend with pods, events, and resource metrics, but add OAuth authentication.

In [ ]:
# Create FastAPI application with realistic Kubernetes data
app = FastAPI(title="Kubernetes API Simulator", version="1.0.0")

# Simple OAuth2 token implementation for demonstration
# In production, you would use a more robust implementation
oauth2_scheme = OAuth2PasswordBearer(tokenUrl="token")
SECRET_KEY = "sre_agent_workshop_secret_key"  # In production, use a secure key
ACCESS_TOKEN_EXPIRE_MINUTES = 30

# Test users (in production, use a proper user database)
USERS = {
    "sre_agent": {
        "username": "sre_agent",
        "hashed_password": "workshop_password",  # In production, use proper hashing
        "role": "agent"
    }
}

# Token database
tokens_db = {}

# Realistic pod data with expanded multi-pod scenario (same as Module 1)
PODS_DATA = {
    "pods": [
        {
            "name": "payment-service-7d4f8-x5m1q",
            "namespace": "production",
            "status": "CrashLoopBackOff",
            "ready": False,
            "restart_count": 15,
            "cpu_usage": "25%",
            "memory_usage": "98%",
            "node": "worker-node-2",
            "last_restart": "2024-01-15T14:24:30Z",
            "containers": [
                {
                    "name": "payment-api",
                    "image": "payment-service:v1.2.3",
                    "status": "Waiting",
                    "reason": "CrashLoopBackOff",
                    "message": "Back-off 5m0s restarting failed container"
                }
            ]
        },
        {
            "name": "payment-service-7d4f8-j9k7l",
            "namespace": "production",
            "status": "Running",
            "ready": True,
            "restart_count": 2,
            "cpu_usage": "78%",
            "memory_usage": "87%",
            "node": "worker-node-1",
            "last_restart": "2024-01-15T12:15:45Z",
            "containers": [
                {
                    "name": "payment-api",
                    "image": "payment-service:v1.2.3",
                    "status": "Running",
                    "reason": "Started",
                    "message": "Container running but approaching memory limits"
                }
            ]
        },
        {
            "name": "user-service-9k2x1-y6n2r",
            "namespace": "production",
            "status": "Running",
            "ready": True,
            "restart_count": 0,
            "cpu_usage": "32%",
            "memory_usage": "64%",
            "node": "worker-node-1",
            "last_restart": None,
            "containers": [
                {
                    "name": "user-api",
                    "image": "user-service:v1.1.0",
                    "status": "Running",
                    "reason": "Started",
                    "message": "Container started successfully"
                }
            ]
        },
        {
            "name": "database-service-3r5t6-h8j9k",
            "namespace": "production",
            "status": "Running",
            "ready": True,
            "restart_count": 0,
            "cpu_usage": "45%",
            "memory_usage": "72%",
            "node": "worker-node-2",
            "last_restart": None,
            "containers": [
                {
                    "name": "postgres",
                    "image": "postgres:14.5",
                    "status": "Running",
                    "reason": "Started",
                    "message": "Container started successfully"
                }
            ]
        }
    ]
}

# Detailed pod events data (same as Module 1)
EVENTS_DATA = {
    "events": {
        "payment-service-7d4f8-x5m1q": [
            {
                "type": "Warning",
                "reason": "OutOfMemoryKilled",
                "message": "Container payment-api was killed due to OOM (Out of Memory). Memory cgroup usage exceeds configured limit.",
                "count": 15,
                "timestamp": "2024-01-15T14:24:28Z"
            },
            {
                "type": "Warning",
                "reason": "BackOff",
                "message": "Back-off restarting failed container payment-api in pod payment-service-7d4f8-x5m1q",
                "count": 12,
                "timestamp": "2024-01-15T14:24:45Z"
            },
            {
                "type": "Normal",
                "reason": "Pulled",
                "message": "Successfully pulled image 'payment-service:v1.2.3'",
                "count": 16,
                "timestamp": "2024-01-15T14:19:20Z"
            },
            {
                "type": "Warning",
                "reason": "FailedMemoryAllocation",
                "message": "Memory allocation failed with error: Cannot allocate memory for heap",
                "count": 8,
                "timestamp": "2024-01-15T14:22:15Z"
            }
        ],
        "payment-service-7d4f8-j9k7l": [
            {
                "type": "Warning",
                "reason": "MemoryPressure",
                "message": "Container payment-api is approaching memory limit. Current usage: 87% of allocated memory.",
                "count": 5,
                "timestamp": "2024-01-15T14:20:05Z"
            },
            {
                "type": "Warning",
                "reason": "HighCPUUsage",
                "message": "Container payment-api is using high CPU: 78% of allocated CPU.",
                "count": 8,
                "timestamp": "2024-01-15T14:15:30Z"
            },
            {
                "type": "Normal",
                "reason": "Started",
                "message": "Started container payment-api",
                "count": 3,
                "timestamp": "2024-01-15T12:15:50Z"
            }
        ],
        "user-service-9k2x1-y6n2r": [
            {
                "type": "Normal",
                "reason": "Pulled",
                "message": "Successfully pulled image 'user-service:v1.1.0'",
                "count": 1,
                "timestamp": "2024-01-15T08:30:15Z"
            },
            {
                "type": "Normal",
                "reason": "Created",
                "message": "Created container user-api",
                "count": 1,
                "timestamp": "2024-01-15T08:30:18Z"
            },
            {
                "type": "Normal",
                "reason": "Started",
                "message": "Started container user-api",
                "count": 1,
                "timestamp": "2024-01-15T08:30:20Z"
            }
        ],
        "database-service-3r5t6-h8j9k": [
            {
                "type": "Normal",
                "reason": "Pulled",
                "message": "Successfully pulled image 'postgres:14.5'",
                "count": 1,
                "timestamp": "2024-01-14T23:15:10Z"
            },
            {
                "type": "Normal",
                "reason": "Created",
                "message": "Created container postgres",
                "count": 1,
                "timestamp": "2024-01-14T23:15:12Z"
            },
            {
                "type": "Normal",
                "reason": "Started",
                "message": "Started container postgres",
                "count": 1,
                "timestamp": "2024-01-14T23:15:15Z"
            }
        ]
    }
}

# Detailed resource metrics data (historical) - same as Module 1
RESOURCES_DATA = {
    "resources": {
        "payment-service-7d4f8-x5m1q": {
            "memory": [
                {"timestamp": "2024-01-15T12:00:00Z", "value": "65%"},
                {"timestamp": "2024-01-15T12:30:00Z", "value": "72%"},
                {"timestamp": "2024-01-15T13:00:00Z", "value": "79%"},
                {"timestamp": "2024-01-15T13:30:00Z", "value": "85%"},
                {"timestamp": "2024-01-15T14:00:00Z", "value": "91%"},
                {"timestamp": "2024-01-15T14:20:00Z", "value": "98%"},
                {"timestamp": "2024-01-15T14:24:30Z", "value": "100%"}
            ],
            "cpu": [
                {"timestamp": "2024-01-15T12:00:00Z", "value": "18%"},
                {"timestamp": "2024-01-15T12:30:00Z", "value": "20%"},
                {"timestamp": "2024-01-15T13:00:00Z", "value": "22%"},
                {"timestamp": "2024-01-15T13:30:00Z", "value": "24%"},
                {"timestamp": "2024-01-15T14:00:00Z", "value": "25%"},
                {"timestamp": "2024-01-15T14:24:30Z", "value": "25%"}
            ],
            "limits": {"memory": "512Mi", "cpu": "500m"},
            "requests": {"memory": "256Mi", "cpu": "250m"}
        },
        "payment-service-7d4f8-j9k7l": {
            "memory": [
                {"timestamp": "2024-01-15T12:00:00Z", "value": "55%"},
                {"timestamp": "2024-01-15T12:30:00Z", "value": "61%"},
                {"timestamp": "2024-01-15T13:00:00Z", "value": "68%"},
                {"timestamp": "2024-01-15T13:30:00Z", "value": "73%"},
                {"timestamp": "2024-01-15T14:00:00Z", "value": "79%"},
                {"timestamp": "2024-01-15T14:20:00Z", "value": "87%"}
            ],
            "cpu": [
                {"timestamp": "2024-01-15T12:00:00Z", "value": "45%"},
                {"timestamp": "2024-01-15T12:30:00Z", "value": "52%"},
                {"timestamp": "2024-01-15T13:00:00Z", "value": "58%"},
                {"timestamp": "2024-01-15T13:30:00Z", "value": "65%"},
                {"timestamp": "2024-01-15T14:00:00Z", "value": "72%"},
                {"timestamp": "2024-01-15T14:20:00Z", "value": "78%"}
            ],
            "limits": {"memory": "512Mi", "cpu": "500m"},
            "requests": {"memory": "256Mi", "cpu": "250m"}
        },
        "user-service-9k2x1-y6n2r": {
            "memory": [
                {"timestamp": "2024-01-15T12:00:00Z", "value": "58%"},
                {"timestamp": "2024-01-15T12:30:00Z", "value": "60%"},
                {"timestamp": "2024-01-15T13:00:00Z", "value": "62%"},
                {"timestamp": "2024-01-15T13:30:00Z", "value": "63%"},
                {"timestamp": "2024-01-15T14:00:00Z", "value": "63%"},
                {"timestamp": "2024-01-15T14:20:00Z", "value": "64%"}
            ],
            "cpu": [
                {"timestamp": "2024-01-15T12:00:00Z", "value": "30%"},
                {"timestamp": "2024-01-15T12:30:00Z", "value": "31%"},
                {"timestamp": "2024-01-15T13:00:00Z", "value": "31%"},
                {"timestamp": "2024-01-15T13:30:00Z", "value": "32%"},
                {"timestamp": "2024-01-15T14:00:00Z", "value": "32%"},
                {"timestamp": "2024-01-15T14:20:00Z", "value": "32%"}
            ],
            "limits": {"memory": "256Mi", "cpu": "250m"},
            "requests": {"memory": "128Mi", "cpu": "100m"}
        },
        "database-service-3r5t6-h8j9k": {
            "memory": [
                {"timestamp": "2024-01-15T12:00:00Z", "value": "70%"},
                {"timestamp": "2024-01-15T12:30:00Z", "value": "70%"},
                {"timestamp": "2024-01-15T13:00:00Z", "value": "71%"},
                {"timestamp": "2024-01-15T13:30:00Z", "value": "71%"},
                {"timestamp": "2024-01-15T14:00:00Z", "value": "72%"},
                {"timestamp": "2024-01-15T14:20:00Z", "value": "72%"}
            ],
            "cpu": [
                {"timestamp": "2024-01-15T12:00:00Z", "value": "40%"},
                {"timestamp": "2024-01-15T12:30:00Z", "value": "42%"},
                {"timestamp": "2024-01-15T13:00:00Z", "value": "44%"},
                {"timestamp": "2024-01-15T13:30:00Z", "value": "44%"},
                {"timestamp": "2024-01-15T14:00:00Z", "value": "45%"},
                {"timestamp": "2024-01-15T14:20:00Z", "value": "45%"}
            ],
            "limits": {"memory": "1Gi", "cpu": "1000m"},
            "requests": {"memory": "512Mi", "cpu": "500m"}
        }
    }
}

In [ ]:
# OAuth authentication endpoints
def authenticate_user(username: str, password: str):
    """Validate username and password (simplified for workshop)"""
    if username not in USERS:
        return False
    user = USERS[username]
    if user["hashed_password"] != password:  # In production, use proper password verification
        return False
    return user

def create_access_token(data: dict):
    """Create a new access token"""
    token_id = str(uuid.uuid4())
    expires = datetime.utcnow() + timedelta(minutes=ACCESS_TOKEN_EXPIRE_MINUTES)
    
    token_data = data.copy()
    token_data.update({"exp": expires.timestamp(), "token_id": token_id})
    
    # Store token in our simple database
    tokens_db[token_id] = token_data
    
    return token_id

async def get_current_user(token: str = Depends(oauth2_scheme)):
    """Validate token and return user (simplified for workshop)"""
    if token not in tokens_db:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Invalid authentication credentials",
            headers={"WWW-Authenticate": "Bearer"},
        )
        
    token_data = tokens_db[token]
    
    # Check if token is expired
    if datetime.utcnow().timestamp() > token_data["exp"]:
        # Remove expired token
        tokens_db.pop(token)
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Token expired",
            headers={"WWW-Authenticate": "Bearer"},
        )
        
    username = token_data["sub"]
    if username not in USERS:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="User not found",
            headers={"WWW-Authenticate": "Bearer"},
        )
        
    return USERS[username]

@app.post("/token")
async def login(form_data: OAuth2PasswordRequestForm = Depends()):
    """OAuth2 compatible token login endpoint"""
    user = authenticate_user(form_data.username, form_data.password)
    if not user:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Incorrect username or password",
            headers={"WWW-Authenticate": "Bearer"},
        )
        
    access_token = create_access_token(data={"sub": user["username"]})
    
    return {"access_token": access_token, "token_type": "bearer"}

# Secured API endpoints
@app.get("/health")
def health_check():
    """Public health check endpoint"""
    return {"status": "healthy", "service": "kubernetes-api"}

@app.get("/pods")
async def get_pods(current_user: dict = Depends(get_current_user)):
    """Get all pods in the cluster (requires authentication)"""
    return PODS_DATA

@app.get("/pods/{pod_name}/events")
async def get_pod_events(pod_name: str, current_user: dict = Depends(get_current_user)):
    """Get detailed events for a specific pod (requires authentication)"""
    if pod_name not in EVENTS_DATA["events"]:
        raise HTTPException(status_code=404, detail=f"Pod {pod_name} not found")
    return {"events": EVENTS_DATA["events"][pod_name]}

@app.get("/pods/{pod_name}/resources")
async def get_pod_resource_metrics(pod_name: str, current_user: dict = Depends(get_current_user)):
    """Get detailed resource metrics for a specific pod (requires authentication)"""
    if pod_name not in RESOURCES_DATA["resources"]:
        raise HTTPException(status_code=404, detail=f"Pod {pod_name} not found")
    return {"resources": RESOURCES_DATA["resources"][pod_name]}

print("✅ Enhanced FastAPI backend created with OAuth authentication")

In [ ]:
# Start the FastAPI server in background
def start_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="error")

server_thread = threading.Thread(target=start_server, daemon=True)
server_thread.start()

# Wait for server startup
time.sleep(3)

# Verify server is running
try:
    response = requests.get("http://127.0.0.1:8000/health", timeout=5)
    if response.status_code == 200:
        print("✅ Backend server running at http://127.0.0.1:8000")
        print(f"   Health status: {response.json()['status']}")
        
        # Test authentication
        print("\nTesting OAuth authentication:")
        try:
            # Try to access protected endpoint without token
            response = requests.get("http://127.0.0.1:8000/pods")
            print(f"   Unauthorized access: {response.status_code} {response.reason}")
            
            # Get token
            response = requests.post(
                "http://127.0.0.1:8000/token",
                data={"username": "sre_agent", "password": "workshop_password"}
            )
            token = response.json()["access_token"]
            print(f"   Token obtained: {token[:10]}...")
            
            # Access with token
            response = requests.get(
                "http://127.0.0.1:8000/pods",
                headers={"Authorization": f"Bearer {token}"}
            )
            print(f"   Authorized access: {response.status_code} {response.reason}")
            print("✅ Authentication working correctly")
        except Exception as e:
            print(f"❌ Authentication test failed: {e}")
    else:
        print(f"❌ Server health check failed: {response.status_code}")
except Exception as e:
    print(f"❌ Cannot connect to server: {e}")

## Step 4: Configure AgentCore Gateway

Now let's create the Gateway configuration for MCP protocol integration.

In [ ]:
# Create Gateway configuration directory
!mkdir -p gateway_config

# Generate an MCP tool configuration file
mcp_tools_config = {
    "tools": [
        {
            "name": "get_pod_status",
            "description": "Get detailed status information for Kubernetes pods in the specified namespace.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "namespace": {
                        "type": "string",
                        "description": "Kubernetes namespace to query (default: production)"
                    }
                }
            },
            "authentication": {
                "type": "oauth2",
                "credentials": {
                    "client_id": "sre_agent",
                    "client_secret": "workshop_password"
                }
            },
            "endpoint": "http://127.0.0.1:8000/pods",
            "method": "GET"
        },
        {
            "name": "get_pod_events",
            "description": "Get detailed events and warnings for a specific Kubernetes pod.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "pod_name": {
                        "type": "string",
                        "description": "Name of the Kubernetes pod to query"
                    }
                },
                "required": ["pod_name"]
            },
            "authentication": {
                "type": "oauth2",
                "credentials": {
                    "client_id": "sre_agent",
                    "client_secret": "workshop_password"
                }
            },
            "endpoint": "http://127.0.0.1:8000/pods/{pod_name}/events",
            "method": "GET"
        },
        {
            "name": "get_pod_resources",
            "description": "Get detailed resource metrics and history for a specific Kubernetes pod.",
            "input_schema": {
                "type": "object",
                "properties": {
                    "pod_name": {
                        "type": "string",
                        "description": "Name of the Kubernetes pod to query"
                    }
                },
                "required": ["pod_name"]
            },
            "authentication": {
                "type": "oauth2",
                "credentials": {
                    "client_id": "sre_agent",
                    "client_secret": "workshop_password"
                }
            },
            "endpoint": "http://127.0.0.1:8000/pods/{pod_name}/resources",
            "method": "GET"
        }
    ]
}

# Save the configuration
with open("gateway_config/mcp_tools.json", "w") as f:
    json.dump(mcp_tools_config, f, indent=2)

# Generate gateway configuration
gateway_config = {
    "name": "sre-agent-gateway",
    "description": "Gateway for SRE Agent with Kubernetes tools",
    "version": "1.0.0",
    "ssl": {
        "enabled": False,  # In production, this should be True with proper certificates
        "cert_path": "",
        "key_path": ""
    },
    "auth": {
        "type": "oauth2",
        "token_url": "http://127.0.0.1:8000/token",
        "client_id": "sre_agent",
        "client_secret": "workshop_password"
    },
    "tools_config_path": "gateway_config/mcp_tools.json",
    "logging": {
        "level": "info",
        "file": "gateway_config/gateway.log"
    }
}

# Save the configuration
with open("gateway_config/gateway_config.json", "w") as f:
    json.dump(gateway_config, f, indent=2)
    
# Show the configuration files
print("✅ Gateway configuration created")
print("\nGateway Configuration:")
!cat gateway_config/gateway_config.json
print("\nMCP Tools Configuration:")
!cat gateway_config/mcp_tools.json | head -n 20
print("...")

## Step 5: Implement Gateway MCP Client

Now, let's create a client that communicates with the Gateway using MCP protocol.

In [ ]:
class McpGatewayClient:
    """Client for communicating with AgentCore Gateway using MCP protocol"""
    
    def __init__(self, gateway_url, client_id, client_secret):
        """Initialize the MCP Gateway client
        
        Args:
            gateway_url (str): URL of the gateway
            client_id (str): OAuth client ID
            client_secret (str): OAuth client secret
        """
        self.gateway_url = gateway_url
        self.client_id = client_id
        self.client_secret = client_secret
        self.token = None
        self.token_expires = 0
        
    def _get_token(self):
        """Get a new OAuth token or use cached one if valid"""
        current_time = time.time()
        
        # Check if token is still valid
        if self.token and current_time < self.token_expires:
            return self.token
            
        # Get a new token
        try:
            response = requests.post(
                f"{self.gateway_url}/token",
                data={
                    "username": self.client_id,
                    "password": self.client_secret,
                    "grant_type": "password"
                }
            )
            response.raise_for_status()
            token_data = response.json()
            
            self.token = token_data["access_token"]
            # Set token expiry (default to 30 minutes if not specified)
            expires_in = token_data.get("expires_in", 1800)
            self.token_expires = current_time + expires_in
            
            return self.token
        except Exception as e:
            raise Exception(f"Failed to get OAuth token: {e}")
    
    def call_tool(self, tool_name, **kwargs):
        """Call a tool through the MCP Gateway
        
        Args:
            tool_name (str): Name of the tool to call
            **kwargs: Tool parameters
            
        Returns:
            dict: Tool response
        """
        try:
            # Get a valid token
            token = self._get_token()
            
            # Format request according to MCP protocol
            mcp_request = {
                "tool": tool_name,
                "parameters": kwargs
            }
            
            # Send request to gateway
            response = requests.post(
                f"{self.gateway_url}/tools/{tool_name}",
                json=mcp_request,
                headers={
                    "Authorization": f"Bearer {token}",
                    "Content-Type": "application/json"
                }
            )
            response.raise_for_status()
            
            # Parse MCP response
            mcp_response = response.json()
            
            if "error" in mcp_response:
                raise Exception(f"Tool execution failed: {mcp_response['error']}")
                
            return mcp_response.get("result", {})
        except Exception as e:
            raise Exception(f"Failed to call tool {tool_name}: {e}")

# For workshop simulation, we'll create a mock gateway client that directly calls our APIs
# In a real deployment, this would connect to an actual AgentCore Gateway
class MockGatewayClient:
    """Simulate Gateway client behavior for workshop purposes"""
    
    def __init__(self, api_url, client_id, client_secret):
        self.api_url = api_url
        self.client_id = client_id
        self.client_secret = client_secret
        self.token = None
        self._get_token()
        
    def _get_token(self):
        """Get authentication token"""
        try:
            response = requests.post(
                f"{self.api_url}/token",
                data={
                    "username": self.client_id, 
                    "password": self.client_secret
                }
            )
            response.raise_for_status()
            self.token = response.json()["access_token"]
            return self.token
        except Exception as e:
            raise Exception(f"Authentication failed: {e}")
    
    def call_tool(self, tool_name, **kwargs):
        """Simulate MCP Gateway tool call"""
        headers = {"Authorization": f"Bearer {self.token}"}
        
        try:
            if tool_name == "get_pod_status":
                response = requests.get(
                    f"{self.api_url}/pods",
                    headers=headers
                )
                response.raise_for_status()
                return self._format_pod_status_response(response.json(), kwargs.get("namespace", "production"))
                
            elif tool_name == "get_pod_events":
                pod_name = kwargs.get("pod_name")
                if not pod_name:
                    raise ValueError("pod_name is required")
                    
                response = requests.get(
                    f"{self.api_url}/pods/{pod_name}/events",
                    headers=headers
                )
                response.raise_for_status()
                return self._format_events_response(pod_name, response.json())
                
            elif tool_name == "get_pod_resources":
                pod_name = kwargs.get("pod_name")
                if not pod_name:
                    raise ValueError("pod_name is required")
                    
                response = requests.get(
                    f"{self.api_url}/pods/{pod_name}/resources",
                    headers=headers
                )
                response.raise_for_status()
                return self._format_resources_response(pod_name, response.json())
                
            else:
                raise ValueError(f"Unknown tool: {tool_name}")
                
        except requests.exceptions.HTTPError as e:
            if e.response.status_code == 401:
                # Token might be expired, try to refresh
                self._get_token()
                return self.call_tool(tool_name, **kwargs)
            raise
            
    def _format_pod_status_response(self, data, namespace):
        """Format pod status response similar to our original tool"""
        filtered_pods = [pod for pod in data["pods"] if pod["namespace"] == namespace]
        
        if not filtered_pods:
            return f"No pods found in namespace '{namespace}'"
        
        result = f"Found {len(filtered_pods)} pods in '{namespace}' namespace:\n\n"
        
        for pod in filtered_pods:
            status_icon = "❌" if not pod["ready"] else "✅"
            result += f"{status_icon} Pod: {pod['name']}\n"
            result += f"   Status: {pod['status']} (Ready: {pod['ready']})\n"
            result += f"   Restarts: {pod['restart_count']}\n"
            result += f"   Resource Usage: CPU {pod['cpu_usage']}, Memory {pod['memory_usage']}\n"
            result += f"   Node: {pod['node']}\n"
            
            if pod.get('last_restart'):
                result += f"   Last Restart: {pod['last_restart']}\n"
            
            # Include container details
            if pod.get('containers'):
                result += f"   Containers:\n"
                for container in pod['containers']:
                    result += f"     - {container['name']}: {container['status']} ({container['reason']})\n"
            
            result += "\n"
            
        return result
    
    def _format_events_response(self, pod_name, data):
        """Format events response similar to our original tool"""
        events = data["events"]
        
        if not events:
            return f"No events found for pod '{pod_name}'"
        
        # Sort events by timestamp (newest first)
        sorted_events = sorted(events, key=lambda x: x["timestamp"], reverse=True)
        
        result = f"Events for pod '{pod_name}' (newest first):\n\n"
        
        for event in sorted_events:
            # Format timestamp nicely
            event_time = event["timestamp"]
            event_type = event["type"]
            event_icon = "⚠️" if event_type == "Warning" else "ℹ️" if event_type == "Normal" else "🔴"
            
            result += f"{event_icon} [{event_time}] {event_type}: {event['reason']}\n"
            result += f"    {event['message']}\n"
            result += f"    Occurred {event['count']} times\n\n"
            
        return result
    
    def _format_resources_response(self, pod_name, data):
        """Format resources response similar to our original tool"""
        resources = data["resources"]
        
        result = f"Resource metrics for pod '{pod_name}':\n\n"
        
        # Add resource limits and requests
        result += "Resource Configuration:\n"
        result += f"  Memory Limit:   {resources['limits']['memory']}\n"
        result += f"  Memory Request: {resources['requests']['memory']}\n"
        result += f"  CPU Limit:      {resources['limits']['cpu']}\n"
        result += f"  CPU Request:    {resources['requests']['cpu']}\n\n"
        
        # Memory usage over time
        result += "Memory Usage History (newest first):\n"
        for entry in sorted(resources["memory"], key=lambda x: x["timestamp"], reverse=True):
            timestamp = entry["timestamp"]
            value = entry["value"]
            result += f"  {timestamp}: {value}\n"
        
        result += "\nCPU Usage History (newest first):\n"
        for entry in sorted(resources["cpu"], key=lambda x: x["timestamp"], reverse=True):
            timestamp = entry["timestamp"]
            value = entry["value"]
            result += f"  {timestamp}: {value}\n"
            
        # Add trend analysis
        memory_values = [int(entry["value"].rstrip("%")) for entry in resources["memory"]]
        cpu_values = [int(entry["value"].rstrip("%")) for entry in resources["cpu"]]
        
        memory_trend = "increasing" if memory_values[-1] > memory_values[0] else "decreasing" if memory_values[-1] < memory_values[0] else "stable"
        cpu_trend = "increasing" if cpu_values[-1] > cpu_values[0] else "decreasing" if cpu_values[-1] < cpu_values[0] else "stable"
        
        result += "\nTrend Analysis:\n"
        result += f"  Memory usage is {memory_trend}\n"
        result += f"  CPU usage is {cpu_trend}\n"
        
        return result

# Initialize the mock gateway client
try:
    gateway_client = MockGatewayClient(
        api_url="http://127.0.0.1:8000",
        client_id="sre_agent",
        client_secret="workshop_password"
    )
    
    print("✅ Gateway client initialized successfully")
    
    # Test the gateway client with a simple call
    print("\nTesting gateway client:")
    
    # Get pod status through the gateway
    pod_status = gateway_client.call_tool("get_pod_status", namespace="production")
    print(f"✅ get_pod_status tool call successful")
    print(f"   Found {pod_status.count('Pod:')} pods in response")
    
except Exception as e:
    print(f"❌ Gateway client initialization failed: {e}")

## Step 6: Create Gateway-Enabled Strands Agent Tools

Now let's create Strands Agent tools that use the Gateway client instead of direct API calls.

In [ ]:
@tool
def get_pod_status(namespace: str = "production") -> str:
    """
    Get detailed status information for Kubernetes pods in the specified namespace.
    
    Args:
        namespace: Kubernetes namespace to query (default: production)
        
    Returns:
        Comprehensive pod status including health, resource usage, and recent events
    """
    try:
        # Use gateway client instead of direct API call
        return gateway_client.call_tool("get_pod_status", namespace=namespace)
        
    except Exception as e:
        return f"Error querying Kubernetes API through gateway: {e}"

@tool
def get_pod_events(pod_name: str) -> str:
    """
    Get detailed events and warnings for a specific Kubernetes pod.
    
    Args:
        pod_name: Name of the Kubernetes pod to query
        
    Returns:
        Chronological list of events with timestamps, types, and messages
    """
    try:
        # Use gateway client instead of direct API call
        return gateway_client.call_tool("get_pod_events", pod_name=pod_name)
        
    except Exception as e:
        return f"Error querying Kubernetes API through gateway: {e}"

@tool
def get_pod_resources(pod_name: str) -> str:
    """
    Get detailed resource metrics and history for a specific Kubernetes pod.
    
    Args:
        pod_name: Name of the Kubernetes pod to query
        
    Returns:
        Historical CPU and memory usage, limits, and requests
    """
    try:
        # Use gateway client instead of direct API call
        return gateway_client.call_tool("get_pod_resources", pod_name=pod_name)
        
    except Exception as e:
        return f"Error querying Kubernetes API through gateway: {e}"

print("✅ Three Strands tool functions created with Gateway integration:")
print("   1. get_pod_status() - Pod health and status overview")
print("   2. get_pod_events() - Detailed event history")
print("   3. get_pod_resources() - Resource usage metrics and trends")

## Step 7: Initialize Strands Agent with Gateway-Enabled Tools

Create a Strands Agent that uses our Gateway-enabled tools.

In [ ]:
# Initialize Bedrock model and Strands Agent
try:
    # Create Bedrock model instance
    model = BedrockModel(model_id="us.anthropic.claude-3-haiku-20240307-v1:0")
    
    # Create Strands Agent with professional SRE system prompt
    agent = Agent(
        model=model,
        tools=[get_pod_status, get_pod_events, get_pod_resources],
        system_prompt="""You are an expert Site Reliability Engineer (SRE) investigating infrastructure issues.

        Your responsibilities:
        1. Use available tools to systematically gather infrastructure data
        2. Analyze the information to identify root causes and contributing factors
        3. Provide specific, actionable recommendations for resolution
        4. Explain your reasoning clearly and concisely
        5. Focus on immediate fixes and preventive measures

        When orchestrating multiple tools:
        - Begin with broad status checks to identify affected services
        - Examine event logs for specific error patterns
        - Analyze resource metrics to identify usage patterns and trends
        - Correlate information across tools to build a complete picture
        - Consider relationships between different services

        Be direct, technical, and solution-focused in your analysis."""
    )
    
    print("✅ Strands Agent initialized successfully with Gateway-enabled tools")
    print(f"   Model: Claude 3 Haiku (us.anthropic.claude-3-haiku-20240307-v1:0)")
    print(f"   Tools: 3 tools available (get_pod_status, get_pod_events, get_pod_resources)")
    print(f"   Framework: Strands Agents with AgentCore Gateway integration")
    
except Exception as e:
    print(f"❌ Failed to initialize Strands Agent: {e}")
    print("\nTroubleshooting steps:")
    print("1. Verify AWS credentials: aws configure list")
    print("2. Check Bedrock access in your AWS region")
    print("3. Ensure Claude 3 Haiku model is enabled")
    agent = None

## Step 8: Run Secure Investigation Through Gateway

Let's run the same investigation as in Module 1, but now through our secure Gateway.

In [ ]:
if agent:
    print("🚨 PRODUCTION INCIDENT")
    print("=" * 40)
    print("ALERT: Payment service degradation detected!")
    print("Impact: Users reporting slow payment processing and some failures")
    print("Priority: P1 - Critical")
    print("\nStarting AI investigation through secure Gateway...\n")
    
    # Record investigation start time
    start_time = time.time()
    
    # Run the investigation
    incident_description = (
        "URGENT: We're seeing degraded performance in our payment service. "
        "Users are reporting that payments are taking a long time to process "
        "and some are failing completely. This started approximately 30 minutes ago "
        "and is getting worse. Please investigate all services in the production "
        "namespace and determine what's causing this issue."
    )
    
    try:
        response = agent(incident_description)
        investigation_time = round(time.time() - start_time, 1)
        
        print(f"⚡ Investigation completed in {investigation_time} seconds")
        print("\n" + "=" * 60)
        print("AI INVESTIGATION RESULTS (VIA SECURE GATEWAY)")
        print("=" * 60)
        
        # Access the response correctly
        if hasattr(response, 'content'):
            print(response.content)
        elif hasattr(response, 'message'):
            print(response.message)
        else:
            print(str(response))
        
        print("\n" + "=" * 60)
        
    except Exception as e:
        print(f"❌ Investigation failed: {e}")
        
else:
    print("❌ Cannot run investigation - Strands Agent not initialized")
    print("Please check the previous steps for errors")

## Step 9: Compare Direct vs. Gateway Architecture

Let's analyze the differences between direct API calls and Gateway-mediated architecture.

In [ ]:
print("🔒 SECURITY ARCHITECTURE COMPARISON")
print("=" * 40)

# Direct API Architecture (Module 1)
print("\nDirect API Architecture (Module 1):")
print("✓ Simple implementation")
print("✓ Lower latency (fewer hops)")
print("✓ Easier local development")
print("❌ No authentication")
print("❌ No standardized protocol")
print("❌ No centralized logging or monitoring")
print("❌ Not production-ready")

# Gateway Architecture (Module 2)
print("\nGateway Architecture (Module 2):")
print("✓ OAuth authentication and authorization")
print("✓ Standardized MCP protocol")
print("✓ Centralized logging and monitoring")
print("✓ Enhanced security")
print("✓ Production-ready")
print("❌ More complex implementation")
print("❌ Slightly higher latency")

# Key Benefits of Gateway Architecture
print("\nKey Benefits of AgentCore Gateway:")
print("1. Security: Authentication, authorization, encryption")
print("2. Standardization: MCP protocol for consistent tool interfaces")
print("3. Scalability: Multiple backends through one gateway")
print("4. Observability: Centralized logging and monitoring")
print("5. Production readiness: Enterprise-grade infrastructure")

# Gateway Architecture Benefits for Multi-Agent Systems
print("\nGateway Benefits for Future Multi-Agent Systems:")
print("• Centralized access control for all agents")
print("• Consistent tool interfaces across agents")
print("• Tool sharing between agents")
print("• Cross-agent monitoring and analytics")
print("• Foundation for memory integration")

## Summary and Next Steps

### What You Accomplished

In this workshop module, you:

1. **Understood AgentCore Gateway architecture** and its security benefits
2. **Implemented OAuth authentication** for secure access
3. **Created Gateway configuration** with MCP protocol integration
4. **Transformed tools** to use the Gateway instead of direct API calls
5. **Ran secure investigations** through the Gateway

### Key Learnings

- **AgentCore Gateway** provides a secure intermediary between agents and backends
- **OAuth authentication** ensures only authorized agents can access tools
- **MCP protocol** standardizes tool interfaces for consistent interaction
- **Gateway architecture** enables production-ready security and scalability
- **Same functionality** can be achieved with enhanced security

### Workshop Progression

This module demonstrated Gateway integration. The complete workshop series continues with:

- **Module 3**: Multi-domain agent with cross-system analysis
- **Module 4**: Multi-agent architecture with specialist agents
- **Module 5**: Memory integration for persistent learning
- **Module 6**: Production deployment to AgentCore Runtime

### Resources

- [AgentCore Gateway Documentation](https://docs.aws.amazon.com/bedrock/latest/userguide/agents.html)
- [Model Context Protocol (MCP) Specification](https://aws.amazon.com/bedrock/)
- [OAuth 2.0 Authorization Framework](https://oauth.net/2/)

---

**Next**: Continue to Module 3 to implement multi-domain agent with cross-system analysis.